In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("PySparkPractice").getOrCreate()

employee_data = [
    (101, "Sravan", "Data Engineer", "IT", 75000, "Hyderabad", 28, "2021-05-10", "Male"),
    (102, "Ravi", "Software Engineer", "IT", 68000, "Bangalore", 30, "2020-03-15", "Male"),
    (103, "Priya", "Data Analyst", "Analytics", 62000, "Chennai", 26, "2022-01-12", "Female"),
    (104, "Kiran", "Manager", "HR", 90000, "Mumbai", 35, "2018-07-19", "Male"),
    (105, "Anjali", "HR Executive", "HR", 45000, "Pune", 24, "2023-02-20", "Female"),
    (106, "Vikram", "Data Scientist", "Analytics", 98000, "Delhi", 32, "2019-11-25", "Male"),
    (107, "Sneha", "Developer", "IT", 71000, "Hyderabad", 27, "2021-08-17", "Female"),
    (108, "Rahul", "Tester", "QA", 55000, "Chennai", 29, "2020-06-10", "Male"),
    (109, "Meena", "QA Lead", "QA", 83000, "Bangalore", 33, "2017-09-14", "Female"),
    (110, "Arjun", "Support Engineer", "Support", 50000, "Pune", 31, "2022-04-11", "Male")
]

columns = ["emp_id","name","designation","department","salary","city","age","joining_date","gender"]

emp_df = spark.createDataFrame(employee_data, columns)

df = emp_df

In [0]:
# Remove Duplicate Records
df.dropDuplicates().show()

+------+------+-----------------+----------+------+---------+---+------------+------+
|emp_id|  name|      designation|department|salary|     city|age|joining_date|gender|
+------+------+-----------------+----------+------+---------+---+------------+------+
|   101|Sravan|    Data Engineer|        IT| 75000|Hyderabad| 28|  2021-05-10|  Male|
|   102|  Ravi|Software Engineer|        IT| 68000|Bangalore| 30|  2020-03-15|  Male|
|   103| Priya|     Data Analyst| Analytics| 62000|  Chennai| 26|  2022-01-12|Female|
|   104| Kiran|          Manager|        HR| 90000|   Mumbai| 35|  2018-07-19|  Male|
|   105|Anjali|     HR Executive|        HR| 45000|     Pune| 24|  2023-02-20|Female|
|   106|Vikram|   Data Scientist| Analytics| 98000|    Delhi| 32|  2019-11-25|  Male|
|   107| Sneha|        Developer|        IT| 71000|Hyderabad| 27|  2021-08-17|Female|
|   108| Rahul|           Tester|        QA| 55000|  Chennai| 29|  2020-06-10|  Male|
|   109| Meena|          QA Lead|        QA| 83000|Ban

In [0]:
# Handle Missing Values
null_data = [
    (101, "Sravan", "IT", 75000),
    (102, None, "IT", 68000),
    (103, "Priya", None, 62000),
    (104, "Kiran", "HR", None)
]

null_columns = ["emp_id","name","department","salary"]

null_df = spark.createDataFrame(
    null_data,
    null_columns
)

null_df.fillna({
    "name":"Unknown",
    "department":"NA",
    "salary":0
}).show()

+------+-------+----------+------+
|emp_id|   name|department|salary|
+------+-------+----------+------+
|   101| Sravan|        IT| 75000|
|   102|Unknown|        IT| 68000|
|   103|  Priya|        NA| 62000|
|   104|  Kiran|        HR|     0|
+------+-------+----------+------+



In [0]:
# Standardize Department Names
df.replace(
    ["IT","HR"],
    ["Technology","Human Resources"],
    "department"
).show()

+------+------+-----------------+---------------+------+---------+---+------------+------+
|emp_id|  name|      designation|     department|salary|     city|age|joining_date|gender|
+------+------+-----------------+---------------+------+---------+---+------------+------+
|   101|Sravan|    Data Engineer|     Technology| 75000|Hyderabad| 28|  2021-05-10|  Male|
|   102|  Ravi|Software Engineer|     Technology| 68000|Bangalore| 30|  2020-03-15|  Male|
|   103| Priya|     Data Analyst|      Analytics| 62000|  Chennai| 26|  2022-01-12|Female|
|   104| Kiran|          Manager|Human Resources| 90000|   Mumbai| 35|  2018-07-19|  Male|
|   105|Anjali|     HR Executive|Human Resources| 45000|     Pune| 24|  2023-02-20|Female|
|   106|Vikram|   Data Scientist|      Analytics| 98000|    Delhi| 32|  2019-11-25|  Male|
|   107| Sneha|        Developer|     Technology| 71000|Hyderabad| 27|  2021-08-17|Female|
|   108| Rahul|           Tester|             QA| 55000|  Chennai| 29|  2020-06-10|  Male|

In [0]:
# Join Employee and Department Data
department_data = [
("IT","John"),
("HR","Smith"),
("QA","David"),
("Analytics","Kevin"),
("Support","Robert")
]

dept_df = spark.createDataFrame(
    department_data,
    ["department","manager"]
)

df.join(
    dept_df,
    "department",
    "inner"
).show()

+----------+------+------+-----------------+------+---------+---+------------+------+-------+
|department|emp_id|  name|      designation|salary|     city|age|joining_date|gender|manager|
+----------+------+------+-----------------+------+---------+---+------------+------+-------+
|        IT|   101|Sravan|    Data Engineer| 75000|Hyderabad| 28|  2021-05-10|  Male|   John|
|        IT|   102|  Ravi|Software Engineer| 68000|Bangalore| 30|  2020-03-15|  Male|   John|
| Analytics|   103| Priya|     Data Analyst| 62000|  Chennai| 26|  2022-01-12|Female|  Kevin|
|        HR|   104| Kiran|          Manager| 90000|   Mumbai| 35|  2018-07-19|  Male|  Smith|
|        HR|   105|Anjali|     HR Executive| 45000|     Pune| 24|  2023-02-20|Female|  Smith|
| Analytics|   106|Vikram|   Data Scientist| 98000|    Delhi| 32|  2019-11-25|  Male|  Kevin|
|        IT|   107| Sneha|        Developer| 71000|Hyderabad| 27|  2021-08-17|Female|   John|
|        QA|   108| Rahul|           Tester| 55000|  Chennai

In [0]:
# Department-wise Employee Summary
df.groupBy("department") \
  .agg(
      count("*").alias("employee_count")
  ) \
  .show()

+----------+--------------+
|department|employee_count|
+----------+--------------+
|        IT|             3|
| Analytics|             2|
|        HR|             2|
|        QA|             2|
|   Support|             1|
+----------+--------------+



In [0]:
# Average Salary by Department
df.groupBy("department") \
  .agg(
      avg("salary").alias("avg_salary")
  ) \
  .show()

+----------+-----------------+
|department|       avg_salary|
+----------+-----------------+
|        IT|71333.33333333333|
| Analytics|          80000.0|
|        HR|          67500.0|
|        QA|          69000.0|
|   Support|          50000.0|
+----------+-----------------+



In [0]:
# Salary Categories
df.withColumn(
    "salary_category",
    when(col("salary") >= 90000,"High")
    .when(col("salary") >= 60000,"Medium")
    .otherwise("Low")
).show()

+------+------+-----------------+----------+------+---------+---+------------+------+---------------+
|emp_id|  name|      designation|department|salary|     city|age|joining_date|gender|salary_category|
+------+------+-----------------+----------+------+---------+---+------------+------+---------------+
|   101|Sravan|    Data Engineer|        IT| 75000|Hyderabad| 28|  2021-05-10|  Male|         Medium|
|   102|  Ravi|Software Engineer|        IT| 68000|Bangalore| 30|  2020-03-15|  Male|         Medium|
|   103| Priya|     Data Analyst| Analytics| 62000|  Chennai| 26|  2022-01-12|Female|         Medium|
|   104| Kiran|          Manager|        HR| 90000|   Mumbai| 35|  2018-07-19|  Male|           High|
|   105|Anjali|     HR Executive|        HR| 45000|     Pune| 24|  2023-02-20|Female|            Low|
|   106|Vikram|   Data Scientist| Analytics| 98000|    Delhi| 32|  2019-11-25|  Male|           High|
|   107| Sneha|        Developer|        IT| 71000|Hyderabad| 27|  2021-08-17|Fema

In [0]:
# Optimize Data Using Repartition
optimized_df = df.repartition("department")

optimized_df.show()

+------+------+-----------------+----------+------+---------+---+------------+------+
|emp_id|  name|      designation|department|salary|     city|age|joining_date|gender|
+------+------+-----------------+----------+------+---------+---+------------+------+
|   101|Sravan|    Data Engineer|        IT| 75000|Hyderabad| 28|  2021-05-10|  Male|
|   102|  Ravi|Software Engineer|        IT| 68000|Bangalore| 30|  2020-03-15|  Male|
|   103| Priya|     Data Analyst| Analytics| 62000|  Chennai| 26|  2022-01-12|Female|
|   104| Kiran|          Manager|        HR| 90000|   Mumbai| 35|  2018-07-19|  Male|
|   105|Anjali|     HR Executive|        HR| 45000|     Pune| 24|  2023-02-20|Female|
|   106|Vikram|   Data Scientist| Analytics| 98000|    Delhi| 32|  2019-11-25|  Male|
|   107| Sneha|        Developer|        IT| 71000|Hyderabad| 27|  2021-08-17|Female|
|   108| Rahul|           Tester|        QA| 55000|  Chennai| 29|  2020-06-10|  Male|
|   109| Meena|          QA Lead|        QA| 83000|Ban

In [0]:
# Cache Frequently Used DataFrames

df.groupBy("department") \
  .avg("salary") \
  .show()

+----------+-----------------+
|department|      avg(salary)|
+----------+-----------------+
|        IT|71333.33333333333|
| Analytics|          80000.0|
|        HR|          67500.0|
|        QA|          69000.0|
|   Support|          50000.0|
+----------+-----------------+



In [0]:
# Final Business Summary Report
df.groupBy("department") \
  .agg(
      count("*").alias("employee_count"),
      avg("salary").alias("avg_salary"),
      max("salary").alias("max_salary"),
      min("salary").alias("min_salary")
  ) \
  .show()

+----------+--------------+-----------------+----------+----------+
|department|employee_count|       avg_salary|max_salary|min_salary|
+----------+--------------+-----------------+----------+----------+
|        IT|             3|71333.33333333333|     75000|     68000|
| Analytics|             2|          80000.0|     98000|     62000|
|        HR|             2|          67500.0|     90000|     45000|
|        QA|             2|          69000.0|     83000|     55000|
|   Support|             1|          50000.0|     50000|     50000|
+----------+--------------+-----------------+----------+----------+

